# 05 - 소비자로 시맨틱 검색

이 Notebook에서는 시맨틱 검색을 사용해 AWS Agent Registry에서 에이전트와 도구를 찾은 다음, 런타임 통합을 위한 연결 메타데이터를 추출하는 **소비자** 페르소나 워크플로를 살펴봅니다.

## 학습 내용

- 카탈로그 탐색 — control plane을 통해 레지스트리와 레코드 나열(읽기 전용)
- 소비자 가드레일 검증 — 소비자가 레코드를 생성, 수정 또는 승인할 수 없는지 확인
- 시맨틱 검색 — 자연어 쿼리를 사용하여 에이전트와 도구 검색
- 필터링된 검색 — 설명자 유형, 이름, 버전으로 결과 범위 좁히기
- 연결 메타데이터 추출 — MCP 서버 스키마, A2A agent card, 사용자 지정 설명자 파싱

## 사전 요구 사항

- boto3 >= 1.42.87
- 관리자, 게시자, 소비자 페르소나용 IAM 역할을 생성하려면 [Notebook 01](01-create-user-personas-workflow.ipynb)을 실행하세요.
- 관리자로 레지스트리를 생성하려면 [Notebook 02](02-creating-registry-workflow.ipynb)를 실행하세요.
- 게시자로 레지스트리에 레코드를 게시하려면 [Notebook 03](03-publishing-records-workflow.ipynb)을 실행하세요.
- 승인 워크플로를 수행하려면 [Notebook 04](04-admin-approval-workflow.ipynb)를 실행하세요.

## 소비자로 시맨틱 검색
![시맨틱 검색 워크플로](images/semantic_search_flow_architecture.png)

## 소비자 API 참조

| # | API | 설명 |
|---|-----|-------------|
| 1 | [ListRegistries](https://docs.aws.amazon.com/boto3/latest/reference/services/bedrock-agentcore-control/client/list_registries.html) | 사용 가능한 레지스트리 탐색(control plane) |
| 2 | [ListRegistryRecords](https://docs.aws.amazon.com/boto3/latest/reference/services/bedrock-agentcore-control/client/list_registry_records.html) | 레지스트리의 레코드 탐색(control plane) |
| 3 | [GetRegistryRecord](https://docs.aws.amazon.com/boto3/latest/reference/services/bedrock-agentcore-control/client/get_registry_record.html) | 전체 레코드 세부 정보 가져오기(control plane) |
| 4 | [SearchRegistryRecords](https://docs.aws.amazon.com/boto3/latest/reference/services/bedrock-agentcore/client/search_registry_records.html) | APPROVED 레코드 시맨틱 검색(data plane) |

### Notebook 진행 순서

02(레지스트리 생성) → 03(레코드 게시) → 04(관리자 승인) → **05(이 Notebook)**

#### 사용 사례: 엔터프라이즈 결제 처리
**소비자 페르소나:** 관리자가 레코드를 승인하면 AnyCompany 엔터프라이즈 전반의 AI 에이전트가 유연한 검색 기능을 통해 승인된 Payment Processing 기능을 검색하고 사용할 수 있습니다. 여기에는 자연어 쿼리("find payment processing tools"), 개념적으로 관련된 기능을 찾는 시맨틱 검색, 정확한 기준 일치를 위한 고급 필터링 연산자(in, ne, or, and)가 포함됩니다. 이를 통해 에이전트는 필요한 결제 처리, 환불 처리, 거래 상태 도구를 신속하게 찾고 통합할 수 있습니다. 사용자 지정 통합 코드 없이 이러한 기능을 원활하게 호출하여 웹 채팅, 모바일 앱, 음성 채널 전반에 결제 기능이 포함된 고객 서비스 환경을 더 빠르게 배포할 수 있습니다.

---
## 1. boto3 SDK 및 종속성 설치

핵심 종속성(`boto3` 및 `python-dotenv`)을 설치합니다.

In [ ]:
!pip install boto3 python-dotenv --force-reinstall

## 2. 소비자로 boto3 Session 초기화

`consumer_persona` IAM 역할을 수임하고 임시 자격 증명으로 boto3 Session을 생성합니다. 소비자는 레지스트리와 레코드에 대한 읽기 전용 액세스 권한을 갖습니다.

In [ ]:
import os
import json
import boto3

import utils

AWS_REGION = os.environ.get("AWS_DEFAULT_REGION", "us-west-2")

# 계정 자동 감지
sts = boto3.client("sts", region_name=AWS_REGION)
ACCOUNT_ID = sts.get_caller_identity()["Account"]

# 소비자 역할 수임
CONSUMER_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/consumer_persona"
creds = utils.assume_role(role_arn=CONSUMER_ROLE_ARN, session_name="consumer-session")

consumer_session = boto3.Session(
    aws_access_key_id=creds["AccessKeyId"],
    aws_secret_access_key=creds["SecretAccessKey"],
    aws_session_token=creds["SessionToken"],
    region_name=AWS_REGION,
)

---
## 3. Control Plane 클라이언트 초기화

Control plane(`bedrock-agentcore-control`)은 레지스트리와 레코드에 대한 CRUD 작업을 처리합니다.

In [ ]:
# Control plane 클라이언트(탐색, 가드레일)
cp_client = consumer_session.client("bedrock-agentcore-control")

## 4. 레지스트리 선택

검색할 기존 READY 레지스트리를 찾습니다. `REGISTRY_ID`를 설정하면 해당 값을 검증합니다. 설정하지 않으면 첫 번째 READY 레지스트리를 선택합니다.

In [ ]:
REGISTRY_ID = ""  # 위 목록에서 직접 선택하려면 이 값을 입력

# REGISTRY_ID가 비어 있으면 list_registries에서 첫 번째 READY 레지스트리를 선택

registry_details = utils.get_or_select_registry(cp_client, REGISTRY_ID, AWS_REGION)
REGISTRY_ID = registry_details[0]
REGISTRY_ARN = registry_details[1]

---
## 5. 기본 시맨틱 검색

`SearchRegistryRecords` API는 자연어를 사용하여 관련 에이전트와 도구를 찾습니다.
레코드 이름, 설명 및 설명자 콘텐츠를 대상으로 일치 여부를 확인합니다.

주요 동작:
- `APPROVED` 레코드만 반환됩니다.
- 결과는 검색 쿼리와의 관련성에 따라 정렬됩니다.
- 이 API는 control plane이 아니라 data plane(`bedrock-agentcore`)에 있습니다.

In [ ]:
# Data plane 클라이언트(시맨틱 검색)
dp_client = consumer_session.client("bedrock-agentcore")

### 간단한 자연어 검색

자연어 쿼리를 사용하여 레코드를 검색합니다. 결과는 관련성에 따라 정렬됩니다.

In [ ]:
results = dp_client.search_registry_records(
    registryIds=[REGISTRY_ARN],
    searchQuery="refund chargeback analysis",
    maxResults=5,
)

print(f"Found {len(results['registryRecords'])} matching records:\n")
for rec in results["registryRecords"]:
    print(f"  [{rec['status']}] {rec['name']}")
    print(f"    Type: {rec['descriptorType']}")
    print(f"    Description: {rec.get('description', 'N/A')}")
    print(f"    Record ID: {rec['recordId']}")

### 여러 쿼리 시도

여러 자연어 쿼리를 실행하여 서로 다른 용어에 대해 시맨틱 일치가 어떻게 작동하는지 확인합니다.

In [ ]:
# 여러 자연어 쿼리로 시맨틱 일치가 작동하는 방식 확인
queries = [
    "process a payment",
    "refund chargeback analytics",
    "credit score risk assessment",
    "detect fraudulent transactions",
    "billing dispute resolution",
    "reconcile payments across systems",
    "loan application processing",
]

for q in queries:
    results = dp_client.search_registry_records(
        registryIds=[REGISTRY_ARN],
        searchQuery=q,
        maxResults=3,
    )
    count = len(results["registryRecords"])
    print(f"🔍 '{q}'  →  {count} result(s)")
    for rec in results["registryRecords"]:
        print(f"     [{rec['descriptorType']}] {rec['name']}")
    print()

---
## 6. 필터링된 검색

`filters` 파라미터는 구조화된 연산자가 포함된 JSON 문서를 받아 결과 범위를 좁힙니다.

### 지원되는 필터 필드
- `descriptorType` — MCP, A2A, CUSTOM, AGENT_SKILLS
- `name` — 정확한 레코드 이름
- `version` — 레코드 버전 문자열

### 지원되는 연산자

| 연산자 | 의미 | 예제 |
|----------|---------|---------|
| `$eq` | 같음 | `{"descriptorType": {"$eq": "MCP"}}` |
| `$ne` | 같지 않음 | `{"descriptorType": {"$ne": "CUSTOM"}}` |
| `$in` | 목록에 포함 | `{"descriptorType": {"$in": ["MCP", "A2A"]}}` |
| `$and` | 논리 AND | `{"$and": [filter1, filter2]}` |
| `$or` | 논리 OR | `{"$or": [filter1, filter2]}` |

In [ ]:
# 필터: MCP 레코드만
results = dp_client.search_registry_records(
    registryIds=[REGISTRY_ARN],
    searchQuery="fraud detection suspicious activity",
    maxResults=10,
    filters={"descriptorType": {"$eq": "MCP"}},
)

print("Filter: descriptorType == MCP")
print(f"Results: {len(results['registryRecords'])}\n")
for rec in results["registryRecords"]:
    print(f"  [{rec['descriptorType']}] {rec['name']}")

### 유형 제외(`$ne`)

`$ne`(같지 않음) 연산자를 사용하여 특정 설명자 유형을 제외합니다.

In [ ]:
# 필터: CUSTOM 레코드 제외
results = dp_client.search_registry_records(
    registryIds=[REGISTRY_ARN],
    searchQuery="credit score lending risk",
    maxResults=10,
    filters={"descriptorType": {"$ne": "CUSTOM"}},
)

print("Filter: descriptorType != CUSTOM")
print(f"Results: {len(results['registryRecords'])}\n")
for rec in results["registryRecords"]:
    print(f"  [{rec['descriptorType']}] {rec['name']}")

### 여러 유형 포함(`$in`)

`$in` 연산자를 사용하여 여러 설명자 유형의 레코드를 일치시킵니다.

In [ ]:
# 필터: MCP 또는 A2A 레코드만
results = dp_client.search_registry_records(
    registryIds=[REGISTRY_ARN],
    searchQuery="billing dispute resolution",
    maxResults=10,
    filters={"descriptorType": {"$in": ["MCP", "A2A"]}},
)

print("Filter: descriptorType IN [MCP, A2A]")
print(f"Results: {len(results['registryRecords'])}\n")
for rec in results["registryRecords"]:
    print(f"  [{rec['descriptorType']}] {rec['name']}")

### 복합 필터(`$and`)

`$and`를 사용하여 여러 조건을 결합합니다. 여기서는 특정 버전의 MCP 레코드를 필터링합니다.

In [ ]:
# 복합 필터: 특정 버전의 MCP 레코드
results = dp_client.search_registry_records(
    registryIds=[REGISTRY_ARN],
    searchQuery="reconcile settlement",
    maxResults=10,
    filters={"$and": [{"descriptorType": {"$eq": "MCP"}}, {"version": {"$eq": "1.0"}}]},
)

print("Filter: descriptorType == MCP AND version == 1.0")
print(f"Results: {len(results['registryRecords'])}\n")
for rec in results["registryRecords"]:
    print(f"  [{rec['descriptorType']}] {rec['name']} (v{rec.get('version', '?')})")

### OR 필터(`$or`)

`$or`를 사용하여 주어진 조건 중 하나라도 충족하는 레코드를 일치시킵니다.

In [ ]:
# OR 필터: A2A 또는 CUSTOM 레코드
results = dp_client.search_registry_records(
    registryIds=[REGISTRY_ARN],
    searchQuery="loan application credit check",
    maxResults=10,
    filters={
        "$or": [
            {"descriptorType": {"$eq": "A2A"}},
            {"descriptorType": {"$eq": "CUSTOM"}},
        ]
    },
)

print("Filter: descriptorType == A2A OR descriptorType == CUSTOM")
print(f"Results: {len(results['registryRecords'])}\n")
for rec in results["registryRecords"]:
    print(f"  [{rec['descriptorType']}] {rec['name']}")

---
## 7. 검색 결과에서 연결 메타데이터 추출

검색 결과에는 소비자가 런타임에 검색된 에이전트 또는 도구와 통합하는 데 필요한 연결 메타데이터인 전체 `descriptors`가 포함됩니다.

| 레코드 유형 | 설명자 경로 | 얻을 수 있는 정보 |
|-------------|----------------|--------------|
| MCP | `descriptors.mcp.server.inlineContent` | 서버 이름, 패키지, 전송 구성 |
| MCP | `descriptors.mcp.tools.inlineContent` | 입력 스키마가 포함된 사용 가능한 도구 |
| A2A | `descriptors.a2a.agentCard.inlineContent` | 에이전트 URL, 스킬, 기능 |
| CUSTOM | `descriptors.custom.inlineContent` | 사용자 정의 엔드포인트/구성 |

In [ ]:
# MCP 연결 메타데이터 검색 및 추출
results = dp_client.search_registry_records(
    registryIds=[REGISTRY_ARN],
    searchQuery="payment",
    maxResults=5,
    filters={"descriptorType": {"$eq": "MCP"}},
)

for rec in results["registryRecords"]:
    print(f"=== MCP Record: {rec['name']} ===\n")

    mcp = rec.get("descriptors", {}).get("mcp", {})

    # 서버 스키마 파싱
    server_raw = mcp.get("server", {}).get("inlineContent", "{}")
    server = json.loads(server_raw) if isinstance(server_raw, str) else server_raw
    print(f"  Server: {server.get('name', 'N/A')}")
    print(f"  Version: {server.get('version', 'N/A')}")
    print(f"  Description: {server.get('description', 'N/A')}")
    for pkg in server.get("packages", []):
        print(f"  Package: {pkg.get('identifier')} ({pkg.get('registryType', 'N/A')})")
        print(f"  Transport: {pkg.get('transport', {}).get('type', 'N/A')}")

    # 도구 스키마 파싱
    tools_raw = mcp.get("tools", {}).get("inlineContent", "{}")
    tools = json.loads(tools_raw) if isinstance(tools_raw, str) else tools_raw
    print(f"\n  Tools ({len(tools.get('tools', []))}):")
    for tool in tools.get("tools", []):
        params = list(tool.get("inputSchema", {}).get("properties", {}).keys())
        print(f"    • {tool['name']}: {tool.get('description', '')}")
        print(f"      Parameters: {params}")
    print()

if not results["registryRecords"]:
    print("No MCP records found. Ensure MCP records are APPROVED in the registry.")

### A2A Agent Card 추출

`descriptors.a2a.agentCard`를 파싱하여 에이전트 URL, 스킬 및 기능을 가져옵니다.

In [ ]:
# A2A 연결 메타데이터 검색 및 추출
results = dp_client.search_registry_records(
    registryIds=[REGISTRY_ARN],
    searchQuery="credit score risk assessment agent",
    maxResults=5,
    filters={"descriptorType": {"$eq": "A2A"}},
)

for rec in results["registryRecords"]:
    print(f"=== A2A Record: {rec['name']} ===\n")

    a2a = rec.get("descriptors", {}).get("a2a", {})
    card_raw = a2a.get("agentCard", {}).get("inlineContent", "{}")
    card = json.loads(card_raw) if isinstance(card_raw, str) else card_raw

    print(f"  Agent: {card.get('name', 'N/A')}")
    print(f"  URL: {card.get('url', 'N/A')}")
    print(f"  Version: {card.get('version', 'N/A')}")
    print(f"  Description: {card.get('description', 'N/A')}")
    print(f"\n  Skills ({len(card.get('skills', []))}):")
    for skill in card.get("skills", []):
        print(f"    • {skill.get('name', skill.get('id', 'N/A'))}: {skill.get('description', '')}")
    print()

if not results["registryRecords"]:
    print("No A2A records found. This is expected if A2A records haven't been created/approved yet.")

### CUSTOM 콘텐츠 추출

사용자 정의 엔드포인트 또는 구성 데이터를 가져오기 위해 `descriptors.custom.inlineContent`를 파싱합니다.

In [ ]:
# CUSTOM 연결 메타데이터 검색 및 추출
results = dp_client.search_registry_records(
    registryIds=[REGISTRY_ARN],
    searchQuery="custom endpoint configuration",
    maxResults=5,
    filters={"descriptorType": {"$eq": "CUSTOM"}},
)

for rec in results["registryRecords"]:
    print(f"=== CUSTOM Record: {rec['name']} ===\n")

    custom = rec.get("descriptors", {}).get("custom", {})
    content_raw = custom.get("inlineContent", "{}")
    content = json.loads(content_raw) if isinstance(content_raw, str) else content_raw

    print("  Content:")
    print(json.dumps(content, indent=4))
    print()

if not results["registryRecords"]:
    print("No CUSTOM records found. This is expected if CUSTOM records haven't been created/approved yet.")

---
## 8. 검색 동작 및 엣지 케이스

경계 조건에서 검색이 어떻게 동작하는지 살펴봅니다.

In [ ]:
# maxResults는 반환할 결과 수를 제어(1~20, 기본값 10)
print("=== Pagination: maxResults ===\n")
for n in [1, 3, 10, 20]:
    results = dp_client.search_registry_records(
        registryIds=[REGISTRY_ARN],
        searchQuery="transaction processing",
        maxResults=n,
    )
    print(f"  maxResults={n:<3} → returned {len(results['registryRecords'])} records")

### 빈 결과 집합

일치하는 항목이 없는 쿼리는 오류 없이 빈 목록을 반환합니다.

In [ ]:
# 아무 항목과도 일치하지 않아야 하는 쿼리
results = dp_client.search_registry_records(
    registryIds=[REGISTRY_ARN],
    searchQuery="quantum teleportation flux capacitor",
    maxResults=5,
)

print("Query: 'quantum teleportation flux capacitor'")
print(f"Results: {len(results['registryRecords'])}\n")
if not results["registryRecords"]:
    print("✅ Empty result set returned (no error) — this is expected behavior.")

---
## 9. 정리(모든 Notebook)

이 섹션에서는 Notebook 01–05에서 생성한 모든 리소스를 정리합니다.
- 모든 레지스트리 레코드 삭제(Notebook 03, 04, 05에서 생성)
- 레지스트리 삭제(Notebook 02에서 생성)
- 세 가지 IAM 페르소나 역할 삭제(Notebook 01에서 생성)

⚠️ 모든 Notebook 사용을 마친 후에만 실행하세요. 기본적으로 모든 코드는 주석 처리되어 있습니다.

### 1단계: 관리자 역할 수임

레지스트리, 레코드 및 IAM 역할을 삭제하려면 관리자 권한이 필요합니다.

In [ ]:
# # 정리를 위해 관리자 역할 수임
# admin_creds = utils.assume_role(
#     role_arn=f"arn:aws:iam::{ACCOUNT_ID}:role/admin_persona",
#     session_name="admin-cleanup-session",
# )
# admin_session = boto3.Session(
#     aws_access_key_id=admin_creds["AccessKeyId"],
#     aws_secret_access_key=admin_creds["SecretAccessKey"],
#     aws_session_token=admin_creds["SessionToken"],
#     region_name=AWS_REGION,
# )
# admin_cp = admin_session.client("bedrock-agentcore-control")
# iam_client = boto3.client("iam", region_name=AWS_REGION)
# print("Admin session ready for cleanup.")

### 2단계: 모든 레지스트리 레코드 삭제

레지스트리의 모든 레코드(Notebook 03 및 04의 MCP, A2A, CUSTOM 레코드)를 삭제합니다.

In [ ]:
# # 레지스트리의 모든 레코드 삭제
# try:
#     records = admin_cp.list_registry_records(registryId=REGISTRY_ID)
#     all_recs = records.get("registryRecords", [])
#     print(f"Found {len(all_recs)} record(s) to delete.\n")
#     for rec in all_recs:
#         try:
#             admin_cp.delete_registry_record(
#                 registryId=REGISTRY_ID, recordId=rec["recordId"]
#             )
#             print(f"  ✅ Deleted: {rec['name']} ({rec['recordId']})")
#         except Exception as e:
#             print(f"  ⚠️  Failed: {rec['name']} — {e}")
# except Exception as e:
#     print(f"Error listing records: {e}")

### 3단계: 레지스트리 삭제

Notebook 02에서 생성한 레지스트리를 삭제합니다. 레지스트리를 삭제하려면 먼저 모든 레코드가 제거되어 있어야 합니다.

In [ ]:
# # 레지스트리 삭제
# import time
# try:
#     # 대기 중인 레코드 삭제가 완료될 때까지 대기
#     print("Waiting 10s for record deletions to propagate...")
#     time.sleep(10)

#     admin_cp.delete_registry(registryId=REGISTRY_ID)
#     print(f"✅ Deleted registry: {REGISTRY_ID}")

#     # 삭제가 완료될 때까지 대기
#     print("Waiting for registry deletion...")
#     for _ in range(20):
#         try:
#             r = admin_cp.get_registry(registryId=REGISTRY_ID)
#             print(f"  Status: {r['status']}")
#             if r["status"] == "DELETE_FAILED":
#                 print(f"  ❌ Delete failed: {r.get('statusReason', 'unknown')}")
#                 break
#             time.sleep(5)
#         except admin_cp.exceptions.ResourceNotFoundException:
#             print("  ✅ Registry deleted successfully.")
#             break
# except admin_cp.exceptions.ResourceNotFoundException:
#     print(f"Registry {REGISTRY_ID} not found — already deleted.")
# except Exception as e:
#     print(f"Error deleting registry: {e}")

### 4단계: IAM 페르소나 역할 삭제

Notebook 01에서 생성한 세 가지 페르소나 역할(`admin_persona`, `publisher_persona`, `consumer_persona`)을 삭제합니다.

In [ ]:
# # IAM 페르소나 역할 삭제
# PERSONA_ROLES = ["admin_persona", "publisher_persona", "consumer_persona"]
# POLICY_NAMES = {"admin_persona": "AdminPolicy", "publisher_persona": "PublisherPolicy", "consumer_persona": "ConsumerPolicy"}

# for role_name in PERSONA_ROLES:
#     print(f"\nCleaning up: {role_name}")
#     try:
#         # 인라인 정책 삭제
#         try:
#             inline = iam_client.list_role_policies(RoleName=role_name)
#             for policy_name in inline.get("PolicyNames", []):
#                 iam_client.delete_role_policy(RoleName=role_name, PolicyName=policy_name)
#                 print(f"  Deleted inline policy: {policy_name}")
#         except iam_client.exceptions.NoSuchEntityException:
#             pass

#         # 관리형 정책 분리
#         try:
#             attached = iam_client.list_attached_role_policies(RoleName=role_name)
#             for policy in attached.get("AttachedPolicies", []):
#                 iam_client.detach_role_policy(RoleName=role_name, PolicyArn=policy["PolicyArn"])
#                 print(f"  Detached: {policy['PolicyArn']}")
#         except iam_client.exceptions.NoSuchEntityException:
#             pass

#         # 역할 삭제
#         iam_client.delete_role(RoleName=role_name)
#         print(f"  ✅ Deleted role: {role_name}")

#     except iam_client.exceptions.NoSuchEntityException:
#         print(f"  Role {role_name} not found — already deleted.")
#     except Exception as e:
#         print(f"  ❌ Error: {e}")

# print("\n🧹 Full cleanup complete.")

---
## 사전 요구 Notebook
- **Notebook 01** — [사용자 페르소나 생성](01-create-user-personas-workflow.ipynb): 관리자, 게시자, 소비자 사용자 페르소나 설정
- **Notebook 02** — [레지스트리 생성](02-creating-registry-workflow.ipynb): 관리자가 레지스트리 생성
- **Notebook 03** — [레코드 게시](03-publishing-records-workflow.ipynb): 게시자로 레코드 게시
- **Notebook 04** — [관리자 승인](04-admin-approval-workflow.ipynb): 관리자 승인 워크플로

## 요약

| 수행한 작업 | 사용한 API | Plane |
|-------------|----------|-------|
| 레지스트리 나열 | `ListRegistries` | Control |
| 레코드 탐색 | `ListRegistryRecords` | Control |
| 레코드 세부 정보 조회 | `GetRegistryRecord` | Control |
| 소비자 가드레일 검증 | 다양한 쓰기 API | Control |
| 시맨틱 검색(자연어) | `SearchRegistryRecords` | Data |
| 필터링된 검색(`$eq`, `$ne`, `$in`, `$and`, `$or`) | `SearchRegistryRecords` | Data |
| MCP 서버/도구 메타데이터 추출 | `descriptors.mcp` 파싱 | — |
| A2A agent card 추출 | `descriptors.a2a` 파싱 | — |
| CUSTOM 콘텐츠 추출 | `descriptors.custom` 파싱 | — |

### 핵심 사항

- **검색은 APPROVED 레코드만 반환합니다.** 승인 워크플로는 레코드가 검색 가능해지기 위한 관문입니다.
- **시맨틱 검색은 이름, 설명 및 설명자 콘텐츠를 대상으로 일치 여부를 확인합니다.** 풍부한 설명자는 검색 가능성을 높입니다.
- **필터는 MongoDB 스타일 연산자**(`$eq`, `$ne`, `$in`, `$and`, `$or`)를 `name`, `descriptorType`, `version`에 사용합니다.
- **소비자는 읽기 전용입니다.** 항목을 생성, 수정, 승인 또는 삭제할 수 없습니다.
- **검색은 최종 일관성을 따릅니다.** 새로 승인된 레코드가 표시되는 데 10~30초가 걸릴 수 있습니다.